# Regresión Lineal Simple Utilizando Datos de Automóviles



## Introducción
En este cuaderno, realizaremos un análisis de regresión lineal  utilizando el conjunto de datos de millas por galón (MPG) de automóviles. El objetivo es predecir la autonomía de un automóvil en función de la potencia en caballos.

## Carga y Preparación de Datos
Primero, importamos las bibliotecas necesarias y cargamos los datos.

In [1]:
# Importamos las bibliotecas necesarias para el análisis
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

Vamos a utilizar un conjunto de datos del repositorio de la Universidad de California, Irvine (UCI ML repository):

[Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg)

In [2]:
# Definimos la URL del conjunto de datos
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"

# Especificamos los nombres de las columnas
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model Year', 'Origin', 'Car Name']

# Cargamos los datos en un DataFrame de pandas
data = pd.read_csv(url, delim_whitespace=True, names = column_names, na_values='?')

# Eliminamos las columnas 'Car Name' y 'Origin' ya que no es relevante para nuestro análisis
data = data.drop('Car Name', axis= 1)

data = data.drop('Origin', axis= 1)

# Mostramos las primeras filas del DataFrame
data

C:\Users\Usuario\AppData\Local\Temp\ipykernel_10448\3342642126.py:8: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(url, delim_whitespace=True, names = column_names, na_values='?')


,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year
0,18.0,8,307.0,130.0,3504.0,12.0,70
1,15.0,8,350.0,165.0,3693.0,11.5,70
2,18.0,8,318.0,150.0,3436.0,11.0,70
3,16.0,8,304.0,150.0,3433.0,12.0,70
4,17.0,8,302.0,140.0,3449.0,10.5,70
...,...,...,...,...,...,...,...
393,27.0,4,140.0,86.0,2790.0,15.6,82
394,44.0,4,97.0,52.0,2130.0,24.6,82
395,32.0,4,135.0,84.0,2295.0,11.6,82
396,28.0,4,120.0,79.0,2625.0,18.6,82


In [3]:
#Colocamos la variable objetivo al final
data = data[['Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model Year', 'MPG']]


## Preprocesamiento de Datos
Preparamos los datos para el análisis, tratando los valores faltantes y seleccionando las columnas relevantes.


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Cylinders     398 non-null    int64  
 1   Displacement  398 non-null    float64
 2   Horsepower    392 non-null    float64
 3   Weight        398 non-null    float64
 4   Acceleration  398 non-null    float64
 5   Model Year    398 non-null    int64  
 6   MPG           398 non-null    float64
dtypes: float64(5), int64(2)
memory usage: 21.9 KB


In [5]:
# Observamos que en la columna Horsepower hay 6 datos faltantes. En esta ocasión, vamos simplemente a eliminar las filas
# con los valores faltantes.

data = data.dropna() # Con esto eliminamos la filas que tienen NA
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392 entries, 0 to 397
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Cylinders     392 non-null    int64  
 1   Displacement  392 non-null    float64
 2   Horsepower    392 non-null    float64
 3   Weight        392 non-null    float64
 4   Acceleration  392 non-null    float64
 5   Model Year    392 non-null    int64  
 6   MPG           392 non-null    float64
dtypes: float64(5), int64(2)
memory usage: 24.5 KB


## Análisis de Regresión Lineal

Primero llevaremos a cabo un modelo de regresión lineal con una sola variable predictora

Modelo lineal:

 $$y =  \beta_0 + \beta_1x + \varepsilon$$




Función a minimizar:

 $$S = \sum_{i=1}^{N} (y_i - (\beta_0 + \beta_1x_i))^2$$

In [6]:
# Seleccionamos 'Horsepower' como variable independiente (predictora) y 'MPG' como variable dependiente (objetivo)
y = data['MPG']
X = data[['Horsepower']] # Como queremos que el algoritmo esté en formato de df ponemos los dos corchetes. 


## División de Datos en Conjuntos de Entrenamiento y Prueba
Dividimos los datos en un conjunto de entrenamiento y otro de prueba para poder evaluar el rendimiento de nuestro modelo.


In [8]:
# Establecemos una semilla para la reproducibilidad
np.random.seed(22)


In [10]:

# Mezclamos los datos de forma aleatoria
shuffled_indices = np.random.permutation(len(data))
shuffled_indices



array([146, 362, 154, 119, 254, 177,  67, 297, 158,  51, 187,  59, 188,
       155, 380, 352, 382,  48, 241, 281, 151,   5, 132, 127,  49, 185,
       249, 167, 333, 329,  56,  19,  31,  22,  80, 108,  86, 265, 337,
       161, 136, 336, 319, 220, 134, 301,  69, 290, 304, 176,  16, 210,
        38,  78, 198, 277, 322, 252, 338, 232, 326, 230, 224, 208, 323,
       130, 117, 372, 308, 315, 266, 349, 353, 145,  82,  89, 235, 218,
        50,  40, 313,  99,  85, 166, 223, 350, 128, 148, 227,  61,  87,
       387, 141, 311, 157, 204, 258, 124, 245, 348, 197, 144, 378, 255,
        83, 229,   1, 272, 310, 279, 330, 306, 334, 320, 174, 201, 217,
       178, 206,  41,  36,  77, 135, 156,  33, 192, 354,  21, 213, 113,
       101,  28,  20,   8,   9, 345, 102, 374, 284, 120,  14, 207, 184,
       358,  35, 276, 236,  65, 271, 203, 285,  63,  91,  37, 115,  97,
       269, 287, 361,  30, 150, 179, 256,  55, 118, 242, 376, 342, 263,
       226, 300, 324, 307, 209, 367,  62, 153, 222, 289, 169, 19

In [12]:
# Calculamos el tamaño del conjunto de prueba (20% de los datos)
test_set_size = int(len(data)*0.2)

# Creamos los índices para los conjuntos de prueba y entrenamiento
test_indices = shuffled_indices[:test_set_size]
train_indices = shuffled_indices[test_set_size:]
test_indices

array([146, 362, 154, 119, 254, 177,  67, 297, 158,  51, 187,  59, 188,
       155, 380, 352, 382,  48, 241, 281, 151,   5, 132, 127,  49, 185,
       249, 167, 333, 329,  56,  19,  31,  22,  80, 108,  86, 265, 337,
       161, 136, 336, 319, 220, 134, 301,  69, 290, 304, 176,  16, 210,
        38,  78, 198, 277, 322, 252, 338, 232, 326, 230, 224, 208, 323,
       130, 117, 372, 308, 315, 266, 349, 353, 145,  82,  89, 235, 218],
      dtype=int32)

In [13]:
# Creamos los conjuntos de prueba y entrenamiento
train_set = data.iloc[train_indices]
test_set = data.iloc[test_indices]

# Seleccionamos 'Horsepower' como variable independiente (predictora) y 'MPG' como variable dependiente (respuesta) para ambos conjuntos
X_train = train_set[['Horsepower']]
y_train = train_set[['MPG']]
X_test = test_set[['Horsepower']]
y_test = test_set[['MPG']]


## Construcción y Entrenamiento del Modelo de Regresión Lineal
Construimos el modelo de regresión lineal y lo entrenamos con el conjunto de entrenamiento.


In [14]:
# Creamos una instancia del modelo de regresión lineal
model = LinearRegression()

# Entrenamos el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [15]:
# Realizamos predicciones utilizando el conjunto de prueba
y_pred = model.predict(X_test)

## Evaluación del Modelo con una variable predictora
Evaluamos el modelo usando el conjunto de prueba y calculamos métricas como el error cuadrático medio (MSE). El MSE se define como:

$$ MSE = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2 $$

donde $y_i$ son los valores observados, $\hat{y}_i$ son los valores predichos por el modelo, y $n$ es el número de observaciones.


In [16]:

# Imprimimos los coeficientes del modelo y el error cuadrático medio
print('Término independiente: beta_0:', model.intercept_)
print('Coeficiente: beta_1:', model.coef_)

mse_1variable = mean_squared_error(y_pred, y_test)
print('Error cuadrático medio: %.2f' % mse_1variable)

Término independiente: beta_0: [40.00611142]
Coeficiente: beta_1: [[-0.15818366]]
Error cuadrático medio: 19.99


## Predicción con cada una de las variables

In [17]:
# Variables predictoras
predictors = ['Cylinders', 'Displacement', 'Horsepower', 'Weight', 'Acceleration', 'Model Year']

# Dividimos los datos en conjuntos de entrenamiento y prueba usando librería
X = data[predictors]
y = data['MPG']
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.2, random_state=0)

# Diccionario para almacenar los MSEs
mse_v  = {}

# Entrenar un modelo para cada variable predictora y calcular el MSE
for predictor in predictors:
    X_train_single = X_train[[predictor]]
    X_test_single = X_test[[predictor]]

    model = LinearRegression()
    model.fit(X_train_single, y_train)

    y_pred = model.predict(X_test_single)
    mse = mean_squared_error(y_pred, y_test)
    mse_values[predictor] = mse

# Mostrar los MSEs
for predictor, mse in mse_values.items():
    print(f"MSE utilizando {predictor}: {mse}")

# Identificar la variable con el menor MSE
min_mse_predictor = min(mse_values, key=mse_values.get)
print(f"\nLa variable que produce el menor MSE es: {min_mse_predictor} con un MSE de {mse_values[min_mse_predictor]}")


NameError: name 'mse_values' is not defined

# Predicción con varias variables

## Evaluación del Modelo con dos variable predictoras


Modelo lineal:

 $$y =  \beta_0 + \beta_1x_1 + \beta_2x_2 + \varepsilon$$




Función a minimizar:

 $$S = \sum_{i=1}^{N} (y_i - (\beta_0 + \beta_1x_{1i} + \beta_2 x_{2i}))^2$$

In [18]:
# Seleccionamos 'Horsepower' como variable independiente (predictora) y 'MPG' como variable dependiente (respuesta) para ambos conjuntos
X_train = train_set[['Horsepower','Weight']]
y_train = train_set['MPG']
X_test = test_set[['Horsepower','Weight']]
y_test = test_set['MPG']

In [19]:
# Creamos una instancia del modelo de regresión lineal
model = LinearRegression()

# Entrenamos el modelo con los datos de entrenamiento
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [20]:
# Realizamos predicciones utilizando el conjunto de prueba
y_pred = model.predict(X_test)

# Imprimimos los coeficientes del modelo y el error cuadrático medio
print('Término independiente: beta_0:', model.intercept_)
print('Coeficientes: beta_1, beta_2:', model.coef_)

mse_2variables = mean_squared_error(y_test, y_pred)

print('Error cuadrático medio: %.2f' % mse_2variables)

Término independiente: beta_0: 46.1799023337641
Coeficientes: beta_1, beta_2: [-0.04559703 -0.00604683]
Error cuadrático medio: 16.17


In [21]:
# Calculamos el porcentaje de mejora en el MSE
mejora_mse = ((mse_2variables - mse_1variable) / mse_1variable) * 100

print(f"Reducción del MSE al usar Horsepower y Weight en lugar de solo Horsepower: {mejora_mse:.2f}%")

Reducción del MSE al usar Horsepower y Weight en lugar de solo Horsepower: -19.10%


## Evaluación introduciendo variables secuencialmente

In [23]:
from itertools import combinations
list(combinations(predictors, 3))
# Esto nos sirve para comprobar como podemos combianr de todas las formas posibles varios datos

[('Cylinders', 'Displacement', 'Horsepower'),
 ('Cylinders', 'Displacement', 'Weight'),
 ('Cylinders', 'Displacement', 'Acceleration'),
 ('Cylinders', 'Displacement', 'Model Year'),
 ('Cylinders', 'Horsepower', 'Weight'),
 ('Cylinders', 'Horsepower', 'Acceleration'),
 ('Cylinders', 'Horsepower', 'Model Year'),
 ('Cylinders', 'Weight', 'Acceleration'),
 ('Cylinders', 'Weight', 'Model Year'),
 ('Cylinders', 'Acceleration', 'Model Year'),
 ('Displacement', 'Horsepower', 'Weight'),
 ('Displacement', 'Horsepower', 'Acceleration'),
 ('Displacement', 'Horsepower', 'Model Year'),
 ('Displacement', 'Weight', 'Acceleration'),
 ('Displacement', 'Weight', 'Model Year'),
 ('Displacement', 'Acceleration', 'Model Year'),
 ('Horsepower', 'Weight', 'Acceleration'),
 ('Horsepower', 'Weight', 'Model Year'),
 ('Horsepower', 'Acceleration', 'Model Year'),
 ('Weight', 'Acceleration', 'Model Year')]

In [40]:
#AHORA VAMOS A ANALIZAR LA EVOLUCIÓN AL IR AÑADIENDO MÁS VARIABLES
from itertools import combinations

# Diccionario para almacenar los MSE medios
mse_medios = {}
disminucion_mse_regresion = {}

for num_vars in range(1, len(predictors) +1):
    combinaciones = list(combinations(predictors, num_vars))
    mse_total = 0

    # Realizar cinco divisiones diferentes de entrenamiento y prueba
    for seed in range(5):
        X_train, X_test, y_train, y_test = train_test_split(X[predictors], y, test_size=0.2, random_state=seed)

        for combinacion in combinaciones:
            X_train_comb = X_train[list(combinacion)]
            X_test_comb = X_test[list(combinacion)]

            model = LinearRegression()
            model.fit(X_train_comb, y_train)

            y_pred = model.predict(X_test_comb)
            mse = mean_squared_error(y_test, y_pred)

            mse_total += mse

    # Calcular el MSE medio para esta cantidad de variables predictoras
    mse_medio = mse_total / (len(combinaciones) * 5)  # Dividir por el número total de modelos entrenados
    mse_medios[num_vars] = mse_medio
    print(f"MSE medio con {num_vars} variables predictoras: {mse_medio}")




MSE medio con 1 variables predictoras: 29.763230281405235
MSE medio con 2 variables predictoras: 20.07151382328688
MSE medio con 3 variables predictoras: 16.678380552326455
MSE medio con 4 variables predictoras: 14.945916014555447
MSE medio con 5 variables predictoras: 13.457516617780794
MSE medio con 6 variables predictoras: 11.974418979715866


In [41]:
# Comparar MSE medios para ver la disminución porcentual
for num_vars in range(2, len(predictors) + 1):
    disminucion_mse = ((mse_medios[num_vars - 1] - mse_medios[num_vars]) / mse_medios[num_vars - 1]) * 100
    print(f"Mejora porcentual del MSE medio al pasar de {num_vars - 1} a {num_vars} variables predictoras: {disminucion_mse:.2f}%")

    # Lo almacenamos para la representación gráfica
    disminucion_mse_regresion[num_vars] = disminucion_mse


Mejora porcentual del MSE medio al pasar de 1 a 2 variables predictoras: 32.56%
Mejora porcentual del MSE medio al pasar de 2 a 3 variables predictoras: 16.91%
Mejora porcentual del MSE medio al pasar de 3 a 4 variables predictoras: 10.39%
Mejora porcentual del MSE medio al pasar de 4 a 5 variables predictoras: 9.96%
Mejora porcentual del MSE medio al pasar de 5 a 6 variables predictoras: 11.02%
